In [2]:
import gradio as gr
import numpy as np
from PIL import Image
import cv2
import os
import zipfile
import tempfile
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import BinaryIoU
from tensorflow.keras import backend as K

# ==========================================
# 1. DEFINE CUSTOM METRICS & LOAD MODELS
# ==========================================

# Define the custom functions used during U-Net training
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

print("Loading models... This might take a minute.")

# Load U-Net (Segmentation)
unet = load_model(
    '../models/brain_tumor_unet.keras', 
    custom_objects={
        'dice_loss': dice_loss, 
        'dice_coef': dice_coef,
        'iou': BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou')
    }
)

# Load ResNet50 (Classification)
# Load pretrained (ResNet50)
import keras
from keras.applications import ResNet50
# Load pretrained (ResNet50)

import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Rebuild model
resnet50 = ResNet50(weights='imagenet', input_shape=(224,224,3), include_top=False)

resnet50.trainable = True

for layer in resnet50.layers[:140]:
    layer.trainable = False

model_classification = Sequential()
model_classification.add(resnet50)
model_classification.add(GlobalAveragePooling2D())
model_classification.add(Dense(256, activation='relu'))
model_classification.add(Dropout(0.5))
model_classification.add(Dense(256, activation='relu'))
model_classification.add(Dense(4, activation='softmax'))
model_classification.load_weights('../models/brain_tumor_classification_weights.h5')

print("Models loaded successfully!")

# Define your actual class names
CLASS_NAMES = ['Meningioma', 'Glioma', 'Pituitary', 'No Tumor']

# ==========================================
# 2. PROCESSING FUNCTIONS
# ==========================================

def create_red_overlay(img_array, mask_array):
    """Blends a semi-transparent red mask over the original MRI."""
    red_mask = np.zeros_like(img_array)
    red_mask[:, :] = [255, 0, 0] # Red in RGB
    
    binary_mask = (mask_array > 0.5).astype(np.uint8)
    overlay = img_array.copy()
    tumor_pixels = binary_mask == 1
    
    # Blend 60% original image, 40% red color
    overlay[tumor_pixels] = cv2.addWeighted(img_array, 0.6, red_mask, 0.4, 0)[tumor_pixels]
    return overlay

def process_uploaded_files(file_paths):
    """Handles both individual images and .zip files, predicts, and saves to state."""
    if not file_paths:
        return [], None, gr.update(maximum=1, value=1), {}
    
    image_files = []
    temp_dir = tempfile.mkdtemp() # Create a temporary folder for zip extraction
    
    # 1. Extract Zip or Collect Images
    for file_path in file_paths:
        if file_path.endswith('.zip'):
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(temp_dir)
                # Find all images inside the extracted zip
                for root, _, files in os.walk(temp_dir):
                    for f in files:
                        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                            image_files.append(os.path.join(root, f))
        else:
            image_files.append(file_path)
            
    if not image_files:
        return [], None, gr.update(maximum=1, value=1), {"No valid images found": 1.0}

    # 2. Load and Preprocess all images
    batch_images = []
    for img_path in image_files:
        img = Image.open(img_path).convert('RGB').resize((224, 224))
        batch_images.append(np.array(img))
        
    # Convert to float32 batch for models
    X_batch = (np.array(batch_images) / 255.0).astype('float32')
    
    # 3. Predict Segmentation & Classification
    mask_probs = unet.predict(X_batch, verbose=0) 
    class_probs = model_classification.predict(X_batch, verbose=0)
    
    # 4. Store results for the UI slider
    processed_data = []
    for i in range(len(image_files)):
        orig_img = batch_images[i]
        pred_mask = mask_probs[i].squeeze() 
        overlaid_img = create_red_overlay(orig_img, pred_mask)
        
        # Format probabilities for Gradio
        probs_dict = {CLASS_NAMES[j]: float(class_probs[i][j]) for j in range(4)}
        
        processed_data.append({
            "original": orig_img,
            "overlaid": overlaid_img,
            "probs": probs_dict
        })
        
    first_img = processed_data[0]["overlaid"]
    first_probs = processed_data[0]["probs"]
    
    # Return: State, Image Display, Slider Update, Label Update
    return processed_data, first_img, gr.update(maximum=len(image_files), value=1), first_probs

def update_view(state, slider_idx, show_mask):
    """Updates image and label instantly when slider or checkbox changes."""
    if not state:
        return None, {}
    
    current_data = state[slider_idx - 1]
    img_to_show = current_data["overlaid"] if show_mask else current_data["original"]
    return img_to_show, current_data["probs"]

# ==========================================
# 3. BUILD THE GRADIO UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Brain Tumor MRI Analysis Dashboard")
    
    app_state = gr.State([])
    
    with gr.Row():
        with gr.Column(scale=2):
            # Notice type="filepath" so we get standard file paths for the zip extractor
            file_input = gr.File(file_count="multiple", type="filepath", label="Upload Multiple Images or a .zip file")
            img_display = gr.Image(label="MRI Scan", interactive=False)
            img_slider = gr.Slider(minimum=1, maximum=1, step=1, value=1, label="Image Navigator")
            
        with gr.Column(scale=1):
            mask_toggle = gr.Checkbox(label="Mask On/Off", value=True)
            clf_display = gr.Label(label="Tumor type predictions", num_top_classes=4)
            
    # --- EVENT LISTENERS ---
    file_input.upload(
        fn=process_uploaded_files,
        inputs=[file_input],
        outputs=[app_state, img_display, img_slider, clf_display]
    )
    
    img_slider.change(fn=update_view, inputs=[app_state, img_slider, mask_toggle], outputs=[img_display, clf_display])
    mask_toggle.change(fn=update_view, inputs=[app_state, img_slider, mask_toggle], outputs=[img_display, clf_display])

# Launch the app!
demo.launch()

Loading models... This might take a minute.
Models loaded successfully!


C:\Users\Pruck\AppData\Local\Temp\ipykernel_14636\2890883348.py:160: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
